In [ ]:
from pathlib import Path
import os
import sys
from tqdm import tqdm
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

# Add src directory to path to import organoid_analysis package
sys.path.insert(0, str(Path('..') / 'src'))

from organoid_analysis.utils_data_analysis import plot_plate_view, get_1st_99th_percentile, merge_csv_files


In [ ]:
# Point to the results directory
results_directory = Path("./results/20260114_T7_2microns/")

# Extract the experiment name from the results directory
experiment_id = results_directory.stem

# Point to the conditions file
df_conditions = pd.read_csv(f"./results/{experiment_id}_conditions.csv")

In [ ]:
# Merge the all the csv files into a single dataframe
df_merged = merge_csv_files(results_directory, df_conditions)

In [ ]:
# Calculate % of cells not in organoids on a per-well basis
# Calculate percent of cells not in organoids on a per-well and per-position basis

# Create processed csvs directory relative to results directory
processed_csvs_dir = results_directory / "processed_csvs"
processed_csvs_dir.mkdir(parents=True, exist_ok=True)

# Calculate % of cells not in organoids on a per-well basis
per_well = (
    df_merged.groupby("well_id")
    .apply(lambda g: (g["organoid"] == 0).sum() / len(g) * 100, include_groups=False)
    .round(2)
    .reset_index(name="%_orphans")
)
print("Percent orphan cells per well_id:")
per_well.to_csv(f"./{processed_csvs_dir}/per_well_orphans.csv", index=False)

# Calculate % of cells not in organoids on a per-well and per-position basis
if "multiposition_id" in df_merged.columns:
    per_pos = (
        df_merged.groupby(["well_id", "multiposition_id"])
        .apply(lambda g: (g["organoid"] == 0).sum() / len(g) * 100, include_groups=False)
        .round(2)
        .reset_index(name="%_orphans")
    )
    per_pos.to_csv(f"./{processed_csvs_dir}/per_pos_orphans.csv", index=False)

# Plot plate view of % of cells not in organoids on a per-well basis
plot_plate_view(per_well, "%_orphans", "Percent of cells not in organoids", "Percent", results_directory, fmt=2, display=True, cmap="magma")

In [ ]:
# Remove cells not in organoids, explore data distribution 
#TODO: Clean outliers using size exclusion after exploring segmentation results, this will modify df_merged and subsequent dataframes

# Filter out cells not present in organoids 
df_merged = df_merged[df_merged["organoid"] != 0]

plt.figure(figsize=(10, 5))
sns.histplot(df_merged['cell_area'], bins=100, color='blue', alpha=0.5, label='cell_area')
sns.histplot(df_merged['membrane_area'], bins=100, color='red', alpha=0.5, label='membrane_area')
plt.legend()
plt.xlabel('Area')
plt.ylabel('Count')
plt.title('Distribution of Cell and Membrane Area')

# Cap the x axis if needed for better visualization
plt.xlim(0, 10000)  

plt.show()

In [ ]:
# Generate csvs for aggregated data (well_id, multiposition_id, and unique_organoid_id)

# After filtering out cells not in organoids, calculate mean of every feature on a per well_id basis 
df_merged_per_well = (
    df_merged.groupby(['well_id', 'simplified']).mean(numeric_only=True)
    .drop(columns=['organoid', 'multiposition_id', 'label'], errors='ignore')
    .reset_index()
)
df_merged_per_well.to_csv(f"./{processed_csvs_dir}/per_well_id.csv", index=False)

# After filtering out cells not in organoids, calculate mean of every feature on a per well_id and multiposition_id basis 
df_merged_per_pos = (
    df_merged.groupby(['well_id', 'multiposition_id', 'simplified']).mean(numeric_only=True)
    .drop(columns=['organoid', 'label'], errors='ignore')
    .reset_index()
)
df_merged_per_pos.to_csv(f"./{processed_csvs_dir}/per_pos_id.csv", index=False)

# Create a unique_organoid_id column by concatenating multiposition_id and organoid
df_merged['unique_organoid_id'] = df_merged['multiposition_id'].astype(str) + '_' + df_merged['organoid'].astype(str)

# Calculate mean of every feature on a per unique_organoid_id basis 
df_merged_organoid = (
    df_merged.groupby(['well_id', 'unique_organoid_id', 'simplified']).mean(numeric_only=True)
    .drop(columns=['organoid', 'multiposition_id', 'label'], errors='ignore')
    .reset_index()
)
df_merged_organoid.to_csv(f"./{processed_csvs_dir}/per_organoid_id.csv", index=False)

In [ ]:
# Define dataframes for plotting (read from processed csvs)
df_merged_per_well = pd.read_csv(f"./{processed_csvs_dir}/per_well_id.csv")
df_merged_per_pos = pd.read_csv(f"./{processed_csvs_dir}/per_pos_id.csv")
df_merged_organoid = pd.read_csv(f"./{processed_csvs_dir}/per_organoid_id.csv")


In [ ]:
# Print all columns in df_merged to check which features to plot
for column in df_merged.columns:
    print(column)

In [ ]:
# Create a list containing all the dataframes
dataframes = [(df_merged_per_well, 'per_well_id'), (df_merged_per_pos, 'per_pos_id'), (df_merged_organoid, 'per_organoid_id'), (df_merged, 'per_cell_id')]

# Create tuples of features to plot for correlation plots
features_to_plot = [('membrane_Occludin_RFP_mean_int', 'membrane_Claudin_FITC_mean_int'),
                    ('membrane_Occludin_RFP_sum_int', 'membrane_Claudin_FITC_sum_int'),
                    ('cell_Occludin_RFP_mean_int', 'cell_Claudin_FITC_mean_int'),
                    ('cell_Occludin_RFP_sum_int', 'cell_Claudin_FITC_sum_int'),
                    ('membrane_CellMask_AF647_mean_int', 'membrane_Occludin_RFP_mean_int'),
                    ('membrane_CellMask_AF647_sum_int', 'membrane_Occludin_RFP_sum_int'),
                    ('cell_CellMask_AF647_mean_int', 'cell_Occludin_RFP_mean_int'),
                    ('cell_CellMask_AF647_sum_int', 'cell_Occludin_RFP_sum_int'),
                    ('organoid_solidity', 'membrane_Claudin_FITC_mean_int'),
                    ('organoid_solidity', 'membrane_Claudin_FITC_sum_int'),
                    ('organoid_solidity', 'cell_Claudin_FITC_mean_int'),
                    ('organoid_solidity', 'cell_Claudin_FITC_sum_int'),
                    ('organoid_solidity', 'membrane_Occludin_RFP_mean_int'),
                    ('organoid_solidity', 'membrane_Occludin_RFP_sum_int'),
                    ('organoid_solidity', 'cell_Occludin_RFP_mean_int'),
                    ('organoid_solidity', 'cell_Occludin_RFP_sum_int'),
                    ('organoid_solidity', 'membrane_CellMask_AF647_mean_int'),
                    ('organoid_solidity', 'membrane_CellMask_AF647_sum_int'),
                    ('organoid_solidity', 'cell_CellMask_AF647_mean_int'),
                    ('organoid_solidity', 'cell_CellMask_AF647_sum_int')
]

In [ ]:
# Plot correlation plots across all aggregated dataframes (per well, per 'cell in each' position, per organoid and per cell)

for dataframe, dataframe_name in tqdm(dataframes):

    for feature_pair in features_to_plot:

        # Define x and y axis by unpacking the tuples in features_to_plot
        x_axis, y_axis = feature_pair

        # Example usage for jointplot and axis scaling
        g = sns.jointplot(
            x= x_axis,
            y= y_axis,
            data=dataframe,
            hue='simplified',
            joint_kws=dict(alpha=0.4, s=20)
        )
        g.figure.suptitle(f"Data aggregation: {dataframe_name}", fontsize=14)
        g.figure.subplots_adjust(top=0.93)

        # --- Save plot ---
        save_dir_full = f"{results_directory}/correlation_plots/{dataframe_name}"
        os.makedirs(save_dir_full, exist_ok=True)
        save_path = os.path.join(save_dir_full, f"{x_axis}_VS_{y_axis}.png")
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        #plt.show() # Remove comment if you want to display results in the notebook 
        plt.close(g.figure)   # close this jointplot's figure

In [ ]:
# Plotting full dataset per cell is too crowded, let's subsample 25% of the data for visualization purposes
# In addition, cap the x and y axis using 1st and 99th percentiles

df_sample = df_merged.sample(frac=0.25, random_state=42)

for feature_pair in tqdm(features_to_plot):

    # Define x and y axis by unpacking the tuples in features_to_plot
    x_axis, y_axis = feature_pair

    g = sns.jointplot(
        x= x_axis,
        y=y_axis,
        data=df_sample,
        hue='simplified',
        joint_kws=dict(alpha=0.4, s=20)
    )

    g.figure.suptitle(f"Data aggregation: 'per_cell_id'. 25% subset", fontsize=14)
    g.figure.subplots_adjust(top=0.93)

    # Cap the x and y axis using 1st and 99th percentiles
    plt.xlim(get_1st_99th_percentile(df_merged[x_axis]))
    plt.ylim(get_1st_99th_percentile(df_merged[y_axis]))

    # --- Save plot ---
    save_dir_full = f"{results_directory}/correlation_plots/per_cell_id/0.25_subset"
    os.makedirs(save_dir_full, exist_ok=True)
    save_path = os.path.join(save_dir_full, f"{x_axis}_VS_{y_axis}.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(g.figure)   # close this jointplot's figure

In [ ]:
# Explore drug combination pairs
drug_combinations = [['Stimulation', 'Drug + stimulation'], ['Veh control', 'Drug + veh control'], ['Stimulation', 'Veh control']]

for dataframe, dataframe_name in tqdm(dataframes):

    for drug_combination in drug_combinations:

        for feature_pair in features_to_plot:
        
            # Keep only "drug_combination" pair
            df_filtered = dataframe[dataframe['simplified'].isin(drug_combination)]

            if dataframe_name == 'per_cell_id':
                # Subsample 25% of the filtered data for plotting
                # This only applies to per_cell_id dataset
                df_sample = df_filtered.sample(frac=0.25, random_state=42)
            
            else:

                df_sample = df_filtered # Full dataset

            ## Define x and y axis by unpacking the tuples in features_to_plot
            x_axis, y_axis = feature_pair

            g = sns.jointplot(
                x=x_axis,
                y=y_axis,
                data=df_sample,
                hue='simplified',
                joint_kws=dict(alpha=0.4, s=20)
            )

            if dataframe_name == 'per_cell_id':
                g.figure.suptitle(f"Data aggregation: {dataframe_name}. 25% subset", fontsize=14)
            else:
                g.figure.suptitle(f"Data aggregation: {dataframe_name}.", fontsize=14)
            g.figure.subplots_adjust(top=0.93)

            if dataframe_name == 'per_cell_id':
                # Axis limits from the filtered data (or use df_merged for full-data scale)
                # This only applies to per_cell_id dataset
                plt.xlim(get_1st_99th_percentile(df_filtered[x_axis]))
                plt.ylim(get_1st_99th_percentile(df_filtered[y_axis]))

            # --- Save plot ---
            save_dir_full = f"{results_directory}/correlation_plots/{dataframe_name}/per_treatment/{drug_combination}"
            os.makedirs(save_dir_full, exist_ok=True)
            save_path = os.path.join(save_dir_full, f"{x_axis}_VS_{y_axis}.png")
            plt.savefig(save_path, dpi=300, bbox_inches="tight")
            plt.close(g.figure)   # close this jointplot's figure

### Signal intensity KDE and violin plots

In [ ]:
# Features to plot for intensity distribution plots
features_to_plot_intensity = ['membrane_Occludin_RFP_mean_int',
                    'membrane_Occludin_RFP_sum_int',
                    'cell_Occludin_RFP_mean_int',
                    'cell_Occludin_RFP_sum_int',
                    'membrane_Claudin_FITC_mean_int',
                    'membrane_Claudin_FITC_sum_int',
                    'cell_Claudin_FITC_mean_int',
                    'cell_Claudin_FITC_sum_int',
                    'membrane_CellMask_AF647_mean_int',
                    'membrane_CellMask_AF647_sum_int',
                    'cell_CellMask_AF647_mean_int',
                    'cell_CellMask_AF647_sum_int',
                    'nuclei_DAPI_mean_int',
                    'nuclei_DAPI_sum_int'
]

In [ ]:
for dataframe, dataframe_name in tqdm(dataframes):

    for feature in features_to_plot_intensity:

        g = sns.kdeplot(
            data=dataframe,
            x=feature,
            hue="simplified",
            fill=True,
            common_norm=False,   # important: keep per-condition densities
            alpha=0.4,
            linewidth=2
        )

        g.figure.suptitle(f"Data aggregation: {dataframe_name}.", fontsize=14)

        plt.xlabel(feature)
        plt.ylabel("Density")
        plt.tight_layout()
        # --- Save plot ---
        save_dir_full = f"{results_directory}/intensity_KDE_plots/{dataframe_name}"
        os.makedirs(save_dir_full, exist_ok=True)
        save_path = os.path.join(save_dir_full, f"{feature}.png")
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close(g.figure)


In [ ]:
for dataframe, dataframe_name in tqdm(dataframes):

    for feature in features_to_plot_intensity:

        fig, ax = plt.subplots(figsize=(6, 4))
        sns.violinplot(
            data=dataframe,
            x="simplified",
            y=feature,
            hue="simplified",
            ax=ax,
        )

        ax.set_title(f"{feature}:{dataframe_name}.", fontsize=12)
        ax.set_ylabel("Density")
        ax.set_xlabel("")
        plt.setp(ax.get_xticklabels(), rotation=15, ha="right")
        plt.tight_layout()

        # --- Save plot ---
        save_dir_full = f"{results_directory}/intensity_violin_plots/{dataframe_name}"
        os.makedirs(save_dir_full, exist_ok=True)
        save_path = os.path.join(save_dir_full, f"{feature}.png")
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close(fig)

### Nuclei morphology and DAPI intensity KDE plots

In [ ]:
nuclei_features = [
    "nuclei_area",
    "nuclei_area_bbox",
    "nuclei_area_convex",
    "nuclei_area_filled",
    "nuclei_axis_major_length",
    "nuclei_axis_minor_length",
    "nuclei_equivalent_diameter_area",
    "nuclei_euler_number",
    "nuclei_extent",
    "nuclei_feret_diameter_max",
    "nuclei_solidity",
    "nuclei_inertia_tensor_eigvals-0",
    "nuclei_inertia_tensor_eigvals-1",
    "nuclei_inertia_tensor_eigvals-2",
    "nuclei_DAPI_mean_int",
    "nuclei_DAPI_min_int",
    "nuclei_DAPI_max_int",
    "nuclei_DAPI_std_int",
    "nuclei_DAPI_max_mean_ratio",
    "nuclei_DAPI_sum_int",
]

In [ ]:
for dataframe, dataframe_name in tqdm(dataframes):

    for feature in nuclei_features:

        g = sns.kdeplot(
            data=dataframe,
            x=feature,
            hue="simplified",
            fill=True,
            common_norm=False,   # important: keep per-condition densities
            alpha=0.4,
            linewidth=2
        )

        g.figure.suptitle(f"Data aggregation: {dataframe_name}.", fontsize=14)

        plt.xlabel(feature)
        plt.ylabel("Density")
        plt.tight_layout()
        # --- Save plot ---
        save_dir_full = f"{results_directory}/nuclei_features/{dataframe_name}"
        os.makedirs(save_dir_full, exist_ok=True)
        save_path = os.path.join(save_dir_full, f"{feature}.png")
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close(g.figure)

### Organoid morphology KDE plots

In [ ]:
organoid_morph_features = [
    "organoid_area",
    "organoid_area_bbox",
    "organoid_area_convex",
    "organoid_area_filled",
    "organoid_axis_major_length",
    "organoid_axis_minor_length",
    "organoid_equivalent_diameter_area",
    "organoid_perimeter",
    "organoid_eccentricity",
    "organoid_euler_number",
    "organoid_extent",
    "organoid_feret_diameter_max",
    "organoid_solidity",
    "organoid_inertia_tensor_eigvals-0",
    "organoid_inertia_tensor_eigvals-1",
]

In [ ]:
for feature in organoid_morph_features:

    g = sns.kdeplot(
        data=df_merged_organoid,
        x=feature,
        hue="simplified",
        fill=True,
        common_norm=False,   # important: keep per-condition densities
        alpha=0.4,
        linewidth=2
    )

    g.figure.suptitle(f"Data aggregation: per_organoid.", fontsize=14)

    plt.xlabel(feature)
    plt.ylabel("Density")
    plt.tight_layout()

    # Cap the x axis using 1st and 99th percentiles
    plt.xlim(get_1st_99th_percentile(df_merged_organoid[feature]))

    # --- Save plot ---
    save_dir_full = f"{results_directory}/organoid_features"
    os.makedirs(save_dir_full, exist_ok=True)
    save_path = os.path.join(save_dir_full, f"{feature}.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(g.figure)

### Organoid and cell numbers (plate view)

In [ ]:
# Calculate total cells on a per-well basis
cells_per_well = (
    df_merged.groupby("well_id")
    .size()
    .reset_index(name="total_cell_nr")
)
print("Total cells per well_id:")
cells_per_well.to_csv(f"./{processed_csvs_dir}/per_well_total_cells.csv", index=False)

# Plot plate view of total cells on a per-well basis
plot_plate_view(cells_per_well, "total_cell_nr", "Total cell numbers (in organoids)", "Cells", results_directory, fmt=2, display=True, cmap="magma")

In [ ]:
# Calculate maximum organoid_id on a per-well basis
organoids_per_well = (
    df_merged.groupby("well_id")["organoid"]
    .max()
    .reset_index(name="max_organoid_id") # organoid(_ids) are 1-based and contiguous so .max() works
)
print("Total organoids per well_id:")
organoids_per_well.to_csv(f"./{processed_csvs_dir}/per_well_total_organoids.csv", index=False)

# Plot plate view of total cells on a per-well basis
plot_plate_view(organoids_per_well, "max_organoid_id", "Total organoid numbers", "Organoids", results_directory, fmt=0, display=True, cmap="magma")

### Cell morphology stats

In [ ]:
cell_morph_features = [
    "cell_area",
    "cell_area_bbox",
    "cell_area_convex",
    "cell_area_filled",
    "cell_axis_major_length",
    "cell_axis_minor_length",
    "cell_equivalent_diameter_area",
    "cell_euler_number",
    "cell_extent",
    "cell_feret_diameter_max",
    "cell_solidity",
    "cell_inertia_tensor_eigvals-0",
    "cell_inertia_tensor_eigvals-1",
    "cell_inertia_tensor_eigvals-2",
]

In [ ]:
for feature in cell_morph_features:

    g = sns.kdeplot(
        data=df_merged,
        x=feature,
        hue="simplified",
        fill=True,
        common_norm=False,   # important: keep per-condition densities
        alpha=0.4,
        linewidth=2
    )

    g.figure.suptitle(f"Data aggregation: per_cell.", fontsize=14)

    plt.xlabel(feature)
    plt.ylabel("Density")
    plt.tight_layout()

    # Cap the x axis using 1st and 99th percentiles
    plt.xlim(get_1st_99th_percentile(df_merged[feature]))

    # --- Save plot ---
    save_dir_full = f"{results_directory}/cell_features"
    os.makedirs(save_dir_full, exist_ok=True)
    save_path = os.path.join(save_dir_full, f"{feature}.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(g.figure)

### Membrane morphology stats

In [ ]:
membrane_morph_stats = [
    "membrane_area",
    "membrane_area_bbox",
    "membrane_area_convex",
    "membrane_area_filled",
    "membrane_axis_major_length",
    "membrane_axis_minor_length",
    "membrane_equivalent_diameter_area",
    "membrane_euler_number",
    "membrane_extent",
    "membrane_feret_diameter_max",
    "membrane_solidity",
    "membrane_inertia_tensor_eigvals-0",
    "membrane_inertia_tensor_eigvals-1",
    "membrane_inertia_tensor_eigvals-2",
]

In [ ]:
for feature in membrane_morph_stats:

    g = sns.kdeplot(
        data=df_merged,
        x=feature,
        hue="simplified",
        fill=True,
        common_norm=False,   # important: keep per-condition densities
        alpha=0.4,
        linewidth=2
    )

    g.figure.suptitle(f"Data aggregation: per_cell.", fontsize=14)

    plt.xlabel(feature)
    plt.ylabel("Density")
    plt.tight_layout()

    # Cap the x axis using 1st and 99th percentiles
    plt.xlim(get_1st_99th_percentile(df_merged[feature]))

    # --- Save plot ---
    save_dir_full = f"{results_directory}/membrane_features"
    os.makedirs(save_dir_full, exist_ok=True)
    save_path = os.path.join(save_dir_full, f"{feature}.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(g.figure)